# Testing scipy linear programming

## Imports and definitions

In [1]:
# Module imports
import sys

import numpy as np
from loguru import logger



In [2]:
# Set up the logger
logger.remove()
logger.add(
    sink=sys.stdout,
    format="<level>{level:<10} | {message}</>",
    level="INFO",
    colorize=True,
)

1

### Behaviors as vectors and notation

Suppose a (2,2,2) routed Bell experiment.

$ p(ab|xyz) \in \mathbb{R}^{32}$ is the observed behavior, assumed to be no-signaling : $$p \in \mathcal{NS}$$

We wish for easy conversion from a vector format (for algebraic operations) to a matrix format (for leggibility) :
$$p=\begin{pmatrix} p_{00|00S} \\ p_{00|01S} \\ \vdots \\ p_{00|00L} \\ \vdots \end{pmatrix}\quad \leftrightarrow \quad p=\begin{pmatrix} p_{00|00}& p_{00|01}& p_{00|10}& p_{00|11}\\ p_{01|00}& p_{01|01}& \dots\\ \vdots & & \ddots \\ & & & p_{11|11} \end{pmatrix}$$
where, to account for both matrices $p(z=S)$ and $p(z=L)$, both cases are represented in two matrices, making $p$ either a column vector in $\mathbb{R}^{32}$ or a third-order tensor of shape $(2,4,4)$.

We defined in the Behavior class (behavior.py) some utility functions.

In [3]:
from behaviors import Behavior

# The maximally mixed state over the experiment space
I = Behavior((1 / 4) * np.ones(32))  # noqa: E741

# The usual (2,2,2) PR box
SR_pr_box = np.array(
    [ 1/2, 1/2, 1/2, 0, 0, 0, 0, 1/2, 0, 0, 0, 1/2, 1/2, 1/2, 1/2, 0]
)

# The PR box in the experiment space : p(ab|xy) is assumed to be
# independent of the value of z
pr_box = Behavior(np.concatenate((SR_pr_box, SR_pr_box), axis=0))

def to_behavior(arr):
    return Behavior(np.concatenate((arr, arr), axis=0))

non_positive_arr = np.array(
    [-1/2,-1/2,-1/2,0,0,0,0,1/2,0,0,0,1/2,1/2,1/2,1/2,0,],
)
non_normalized_arr = np.array(
    [1/2,1/2,1/2,0,0,0,0,0,0,0,0,1/2,1/2,1/2,1/2,0,]
)
non_ns_arr = np.array(
    [1/2,1/2,0,0,0,0,1/2,1/2,1/2,1/2,0,0,0,0,1/2,1/2,]
)

non_positive = to_behavior(non_positive_arr)
non_normalized = to_behavior(non_normalized_arr)
non_ns = to_behavior(non_ns_arr)


In [4]:
# Sanity check cell, don't mind me

# print("PR box: ", pr_box)
# print("I: ", I)

print("I == I: ", I == I)
print("I == pr_box: ", I == pr_box)
print("I == 0: ", I == 0)
print("I == 0.25: ", I == 0.25)
print("pr_box == pr_box_array: ", pr_box == np.concatenate((SR_pr_box, SR_pr_box), axis=0))

print("I positive: ", I.positivity())
print("I normalized: ", I.normalization())
print("I no-signaling: ", I.no_signaling())

print("PR box positive: ", pr_box.positivity())
print("PR box normalized: ", pr_box.normalization())
print("PR box no-signaling: ", pr_box.no_signaling())

print("Non-positive: ", non_positive.positivity())
print("Non-normalized: ", non_normalized.normalization())
print("Non-no-signaling: ", non_ns.no_signaling())

print("Non-no-signaling is normalized: ", non_ns.is_normalized())

I == I:  True
I == pr_box:  False
I == 0:  False
I == 0.25:  True
pr_box == pr_box_array:  True
I positive:  True
I normalized:  True
I no-signaling:  True
PR box positive:  True
PR box normalized:  True
PR box no-signaling:  True
Non-positive:  False
Non-normalized:  False
Non-no-signaling:  False
Non-no-signaling is normalized:  True


### Sampling behaviors

In [5]:
from samplers import UniformNormalizedSampler

sampler = UniformNormalizedSampler(2,2,True) # Samples normalized behaviors

sample_behavior = sampler.sample()
print("Sampled behavior: ", sample_behavior)
print("Sample behavior is normalized: ", sample_behavior.is_normalized())
print("Sample behavior is no-signaling: ", sample_behavior.no_signaling())

Sampled behavior:  Behavior:
Short path (z=S):
[[0.22390288 0.45260239 0.00137815 0.09240255]
 [0.0946163  0.04528587 0.10290573 0.18311122]
 [0.00173419 0.16295433 0.80540452 0.46672565]
 [0.67974663 0.33915741 0.09031159 0.25776057]]
Long path (z=L) :
[[0.83466935 0.31752213 0.15893924 0.29893716]
 [0.13794754 0.02041468 0.34263061 0.43736569]
 [0.0131467  0.16753168 0.01448892 0.1684539 ]
 [0.01423641 0.49453151 0.48394124 0.09524325]]
------------
Sample behavior is normalized:  True
Sample behavior is no-signaling:  False


Sampling no-signaling behaviors can't reasonably be achieved with rejection sampling from normalized behaviors though, since as $dim(\mathcal{NS}) < dim(\mathcal{B})$, the usual measure of $\mathcal{NS}$ in $\mathcal{B}$ is null. Getting a no-signaling behavior would thus be very, very lucky.

A consequence of this is that we will need to implement uniform sampling on the $\mathcal{NS}$ polytope to sample no-signaling behaviors directly. This can be achieved if we know the polytope's vertices, which is the case in low-dimensions.